# 📘 Notebook 4 — Pretraining

**MiniMind Step-by-Step Series** | Milestone 4 of 6

In Notebooks 1-3, we built the tokenizer, model, and data pipeline. Now we train the model! We'll implement a complete pretraining loop with mixed-precision training, cosine learning rate scheduling, gradient clipping, and checkpoint saving.

**Learning Objectives:**
- Set up optimizer (AdamW) and cosine learning rate scheduler
- Implement mixed-precision training with AMP (Automatic Mixed Precision)
- Build a training loop with gradient accumulation and clipping
- Track and visualize training loss
- Save and load model checkpoints
- Generate sample text from the trained model

In [ ]:
# === Recap: Setup + Model + Tokenizer + Data Pipeline (from Notebooks 1-3) ===
!pip install tokenizers torch transformers --quiet

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import json
import os
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader

print(f'PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- Config ---
@dataclass
class MiniMindConfig:
    hidden_size: int = 768
    num_hidden_layers: int = 8
    num_attention_heads: int = 8
    num_key_value_heads: int = 4
    head_dim: int = 96
    vocab_size: int = 6400
    intermediate_size: int = 2048
    max_position_embeddings: int = 32768
    rms_norm_eps: float = 1e-6
    rope_theta: float = 1e6
    dropout: float = 0.0
    hidden_act: str = 'silu'
    flash_attn: bool = True
    bos_token_id: int = 1
    eos_token_id: int = 2
    use_moe: bool = False              # Enable Mixture of Experts
    num_experts: int = 4
    num_experts_per_tok: int = 1
    moe_intermediate_size: int = 2048
    norm_topk_prob: bool = True
    router_aux_loss_coef: float = 5e-4

# --- RMSNorm ---
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
    def forward(self, x):
        return (self.weight * self.norm(x.float())).type_as(x)

# --- RoPE ---
def precompute_freqs_cis(dim, end=32768, rope_base=1e6):
    freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2)[:(dim // 2)].float() / dim))
    t = torch.arange(end)
    freqs = torch.outer(t, freqs).float()
    freqs_cos = torch.cat([torch.cos(freqs), torch.cos(freqs)], dim=-1)
    freqs_sin = torch.cat([torch.sin(freqs), torch.sin(freqs)], dim=-1)
    return freqs_cos, freqs_sin

def apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):
    def rotate_half(x):
        return torch.cat((-x[..., x.shape[-1] // 2:], x[..., :x.shape[-1] // 2]), dim=-1)
    cos, sin = cos.unsqueeze(unsqueeze_dim), sin.unsqueeze(unsqueeze_dim)
    return ((q * cos) + (rotate_half(q) * sin)).to(q.dtype), ((k * cos) + (rotate_half(k) * sin)).to(k.dtype)

# --- Attention ---
def repeat_kv(x, n_rep):
    bs, slen, n_kv_heads, head_dim = x.shape
    if n_rep == 1: return x
    return x[:, :, :, None, :].expand(bs, slen, n_kv_heads, n_rep, head_dim).reshape(bs, slen, n_kv_heads * n_rep, head_dim)

class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_local_heads = config.num_attention_heads
        self.n_local_kv_heads = config.num_key_value_heads
        self.n_rep = self.n_local_heads // self.n_local_kv_heads
        self.head_dim = config.head_dim
        self.q_proj = nn.Linear(config.hidden_size, config.num_attention_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, config.num_key_value_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, config.num_key_value_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.num_attention_heads * self.head_dim, config.hidden_size, bias=False)
        self.q_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.k_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.dropout = config.dropout
        self.flash = hasattr(F, 'scaled_dot_product_attention') and config.flash_attn

    def forward(self, x, position_embeddings, past_key_value=None, use_cache=False, attention_mask=None):
        bsz, seq_len, _ = x.shape
        xq, xk, xv = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        xq = xq.view(bsz, seq_len, self.n_local_heads, self.head_dim)
        xk = xk.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)
        xv = xv.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)
        xq, xk = self.q_norm(xq), self.k_norm(xk)
        cos, sin = position_embeddings
        xq, xk = apply_rotary_pos_emb(xq, xk, cos, sin)
        if past_key_value is not None:
            xk = torch.cat([past_key_value[0], xk], dim=1)
            xv = torch.cat([past_key_value[1], xv], dim=1)
        past_kv = (xk, xv) if use_cache else None
        xq, xk, xv = xq.transpose(1, 2), repeat_kv(xk, self.n_rep).transpose(1, 2), repeat_kv(xv, self.n_rep).transpose(1, 2)
        if self.flash and seq_len > 1 and past_key_value is None:
            output = F.scaled_dot_product_attention(xq, xk, xv, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        else:
            scores = (xq @ xk.transpose(-2, -1)) / math.sqrt(self.head_dim)
            scores[:, :, :, -seq_len:] += torch.full((seq_len, seq_len), float("-inf"), device=scores.device).triu(1)
            if attention_mask is not None:
                scores += (1.0 - attention_mask.unsqueeze(1).unsqueeze(2)) * -1e9
            output = self.attn_dropout(F.softmax(scores.float(), dim=-1).type_as(xq)) @ xv
        output = output.transpose(1, 2).reshape(bsz, seq_len, -1)
        return self.resid_dropout(self.o_proj(output)), past_kv

# --- FeedForward (SwiGLU) ---
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

# --- MOE FeedForward ---
class MOEFeedForward(nn.Module):
    """Mixture of Experts: route each token to top-k experts via a learned gate."""
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.gate = nn.Linear(config.hidden_size, config.num_experts, bias=False)
        self.experts = nn.ModuleList([FeedForward(config) for _ in range(config.num_experts)])
    def forward(self, x):
        batch_size, seq_len, hidden_dim = x.shape
        x_flat = x.view(-1, hidden_dim)
        scores = F.softmax(self.gate(x_flat), dim=-1)
        topk_weight, topk_idx = torch.topk(scores, k=self.config.num_experts_per_tok, dim=-1, sorted=False)
        if self.config.norm_topk_prob:
            topk_weight = topk_weight / (topk_weight.sum(dim=-1, keepdim=True) + 1e-20)
        y = torch.zeros_like(x_flat)
        for i, expert in enumerate(self.experts):
            mask = (topk_idx == i)
            if mask.any():
                token_idx = mask.any(dim=-1).nonzero().flatten()
                weight = topk_weight[mask].view(-1, 1)
                y.index_add_(0, token_idx, (expert(x_flat[token_idx]) * weight).to(y.dtype))
            elif self.training:
                y[0, 0] += 0 * sum(p.sum() for p in expert.parameters())
        if self.training and self.config.router_aux_loss_coef > 0:
            load = F.one_hot(topk_idx, self.config.num_experts).float().mean(0)
            self.aux_loss = (load * scores.mean(0)).sum() * self.config.num_experts * self.config.router_aux_loss_coef
        else:
            self.aux_loss = scores.new_zeros(1).squeeze()
        return y.view(batch_size, seq_len, hidden_dim)

# --- Block ---
class MiniMindBlock(nn.Module):
    def __init__(self, layer_id, config):
        super().__init__()
        self.self_attn = Attention(config)
        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.mlp = FeedForward(config) if not config.use_moe else MOEFeedForward(config)
    def forward(self, hidden_states, position_embeddings, past_key_value=None, use_cache=False, attention_mask=None):
        residual = hidden_states
        hidden_states, present_kv = self.self_attn(self.input_layernorm(hidden_states), position_embeddings, past_key_value, use_cache, attention_mask)
        hidden_states = hidden_states + residual
        hidden_states = hidden_states + self.mlp(self.post_attention_layernorm(hidden_states))
        return hidden_states, present_kv

# --- Full Model ---
class MiniMindModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.dropout = nn.Dropout(config.dropout)
        self.layers = nn.ModuleList([MiniMindBlock(l, config) for l in range(config.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        freqs_cos, freqs_sin = precompute_freqs_cis(dim=config.head_dim, end=config.max_position_embeddings, rope_base=config.rope_theta)
        self.register_buffer("freqs_cos", freqs_cos, persistent=False)
        self.register_buffer("freqs_sin", freqs_sin, persistent=False)
    def forward(self, input_ids, attention_mask=None, past_key_values=None, use_cache=False):
        batch_size, seq_length = input_ids.shape
        past_key_values = past_key_values or [None] * len(self.layers)
        start_pos = past_key_values[0][0].shape[1] if past_key_values[0] is not None else 0
        hidden_states = self.dropout(self.embed_tokens(input_ids))
        position_embeddings = (self.freqs_cos[start_pos:start_pos + seq_length], self.freqs_sin[start_pos:start_pos + seq_length])
        presents = []
        for layer, past_kv in zip(self.layers, past_key_values):
            hidden_states, present = layer(hidden_states, position_embeddings, past_key_value=past_kv, use_cache=use_cache, attention_mask=attention_mask)
            presents.append(present)
        hidden_states = self.norm(hidden_states)
        aux_loss = sum([l.mlp.aux_loss for l in self.layers if isinstance(l.mlp, MOEFeedForward)], hidden_states.new_zeros(1).squeeze())
        return hidden_states, presents, aux_loss

class MiniMindForCausalLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.model = MiniMindModel(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.model.embed_tokens.weight = self.lm_head.weight
    def forward(self, input_ids, attention_mask=None, labels=None, past_key_values=None, use_cache=False):
        hidden_states, past_key_values, aux_loss = self.model(input_ids, attention_mask, past_key_values, use_cache)
        logits = self.lm_head(hidden_states)
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), ignore_index=-100)
        return {'loss': loss, 'logits': logits, 'past_key_values': past_key_values, 'aux_loss': aux_loss}

# --- Synthetic Data ---
pretrain_texts = [
    "The transformer architecture was introduced in 2017.",
    "Language models learn to predict the next token in a sequence.",
    "Attention mechanisms allow models to focus on relevant parts of the input.",
    "Neural networks consist of layers of interconnected nodes.",
    "Gradient descent is the primary optimization algorithm for training neural networks.",
    "Backpropagation computes gradients by applying the chain rule.",
    "Tokenization converts raw text into numerical representations.",
    "The softmax function converts logits into probability distributions.",
    "Residual connections help train very deep networks.",
    "Layer normalization stabilizes training by normalizing activations.",
    "Dropout randomly zeroes activations to prevent overfitting.",
    "The learning rate controls the step size during optimization.",
    "Batch size determines how many samples are processed together.",
    "Cross-entropy loss measures the difference between predicted and true distributions.",
    "Weight decay adds L2 regularization to prevent overfitting.",
    "Mixed precision training uses float16 to speed up computation.",
    "The Adam optimizer combines momentum and adaptive learning rates.",
    "Cosine annealing gradually reduces the learning rate during training.",
    "Embedding layers map discrete tokens to continuous vectors.",
    "Position embeddings encode the order of tokens in a sequence.",
]

os.makedirs('synthetic_data', exist_ok=True)
pretrain_path = 'synthetic_data/pretrain.jsonl'
with open(pretrain_path, 'w') as f:
    for text in pretrain_texts:
        f.write(json.dumps({'text': text}) + '\n')

sft_conversations = [
    [{"role": "user", "content": "What is a transformer?"}, {"role": "assistant", "content": "A transformer is a neural network architecture that uses self-attention mechanisms to process sequential data in parallel, rather than sequentially like RNNs."}],
    [{"role": "user", "content": "Explain attention in neural networks."}, {"role": "assistant", "content": "Attention allows a model to dynamically focus on different parts of the input when producing each output element. It computes weighted sums of values based on query-key similarity scores."}],
    [{"role": "user", "content": "What is tokenization?"}, {"role": "assistant", "content": "Tokenization is the process of breaking text into smaller units called tokens. Common methods include BPE (Byte-Pair Encoding), which iteratively merges frequent character pairs."}],
    [{"role": "user", "content": "How does gradient descent work?"}, {"role": "assistant", "content": "Gradient descent updates model parameters by moving them in the direction that reduces the loss function. The step size is controlled by the learning rate."}],
    [{"role": "user", "content": "What is overfitting?"}, {"role": "assistant", "content": "Overfitting occurs when a model learns the training data too well, including noise and outliers, and fails to generalize to new unseen data. Techniques like dropout and weight decay help prevent it."}],
    [{"role": "user", "content": "Explain the softmax function."}, {"role": "assistant", "content": "Softmax converts a vector of real numbers into a probability distribution. Each output value is between 0 and 1, and all outputs sum to 1. It is commonly used in the final layer of classification models."}],
    [{"role": "user", "content": "What is transfer learning?"}, {"role": "assistant", "content": "Transfer learning involves taking a model trained on one task and adapting it to a different but related task. This saves computation and often improves performance, especially with limited data."}],
    [{"role": "user", "content": "What is a loss function?"}, {"role": "assistant", "content": "A loss function measures how far the model's predictions are from the true values. Common examples include cross-entropy for classification and mean squared error for regression."}],
]

sft_path = 'synthetic_data/sft.jsonl'
with open(sft_path, 'w') as f:
    for conv in sft_conversations:
        f.write(json.dumps({'conversations': conv}) + '\n')

# --- Tokenizer ---
from tokenizers import decoders, models, pre_tokenizers, trainers, Tokenizer

VOCAB_SIZE = 6400
all_special = [
    '<|endoftext|>', '<|im_start|>', '<|im_end|>', '<|im_sep|>',
    '<|endofprefix|>', '<|startoftext|>', '<|extra_0|>', '<|extra_1|>',
    '<|extra_2|>', '<|extra_3|>', '<|extra_4|>', '<|extra_5|>',
    '<|extra_6|>', '<|extra_7|>', '<|extra_8|>', '<|extra_9|>',
    '<pad>', '<mask>', '<sep>', '<cls>',
    '<unused_0>', '<unused_1>', '<unused_2>', '<unused_3>',
    '<unused_4>', '<unused_5>', '<unused_6>', '<unused_7>',
    '<think>', '</think>', '<tool_call>', '</tool_call>',
    '<tool_response>', '</tool_response>', '<code>', '</code>',
]

with open(pretrain_path) as f:
    corpus = [json.loads(line)['text'] for line in f] * 100

raw_tokenizer = Tokenizer(models.BPE())
raw_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
trainer_obj = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=all_special,
    show_progress=False,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
)
raw_tokenizer.decoder = decoders.ByteLevel()
raw_tokenizer.train_from_iterator(corpus, trainer=trainer_obj)

class SimpleTokenizer:
    def __init__(self, tokenizer, special_tokens):
        self.tokenizer = tokenizer
        vocab = tokenizer.get_vocab()
        self.bos_token_id = vocab.get('<|im_start|>', 1)
        self.eos_token_id = vocab.get('<|im_end|>', 2)
        self.pad_token_id = vocab.get('<pad>', 0)
    def __call__(self, text, add_special_tokens=False, max_length=None, truncation=False):
        ids = self.tokenizer.encode(text).ids
        if max_length and truncation:
            ids = ids[:max_length]
        return type('Encoding', (), {'input_ids': ids})()
    def decode(self, ids):
        return self.tokenizer.decode(ids)

tokenizer = SimpleTokenizer(raw_tokenizer, all_special)

# --- Chat Template ---
def apply_chat_template(messages):
    text = ''
    for msg in messages:
        text += f'<|im_start|>{msg["role"]}\n{msg["content"]}<|im_end|>\n'
    return text

# --- PretrainDataset ---
class PretrainDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_length=128):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = []
        with open(data_path) as f:
            for line in f:
                self.samples.append(json.loads(line))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, index):
        text = str(self.samples[index]['text'])
        tokens = self.tokenizer(text, add_special_tokens=False, max_length=self.max_length - 2, truncation=True).input_ids
        tokens = [self.tokenizer.bos_token_id] + tokens + [self.tokenizer.eos_token_id]
        input_ids = tokens + [self.tokenizer.pad_token_id] * (self.max_length - len(tokens))
        input_ids = torch.tensor(input_ids[:self.max_length], dtype=torch.long)
        labels = input_ids.clone()
        labels[input_ids == self.tokenizer.pad_token_id] = -100
        return input_ids, labels

# --- SFTDataset ---
class SFTDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, max_length=256):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = []
        with open(jsonl_path) as f:
            for line in f:
                self.samples.append(json.loads(line))
        self.bos_marker = tokenizer(f'<|im_start|>assistant\n', add_special_tokens=False).input_ids
        self.eos_marker = tokenizer(f'<|im_end|>\n', add_special_tokens=False).input_ids
    def __len__(self):
        return len(self.samples)
    def generate_labels(self, input_ids):
        labels = [-100] * len(input_ids)
        i = 0
        while i < len(input_ids):
            if input_ids[i:i + len(self.bos_marker)] == self.bos_marker:
                start = i + len(self.bos_marker)
                end = start
                while end < len(input_ids):
                    if input_ids[end:end + len(self.eos_marker)] == self.eos_marker:
                        break
                    end += 1
                for j in range(start, min(end + len(self.eos_marker), len(input_ids))):
                    labels[j] = input_ids[j]
                i = end + len(self.eos_marker) if end < len(input_ids) else len(input_ids)
            else:
                i += 1
        return labels
    def __getitem__(self, index):
        conversations = self.samples[index]['conversations']
        prompt = apply_chat_template(conversations)
        input_ids = self.tokenizer(prompt).input_ids[:self.max_length]
        input_ids += [self.tokenizer.pad_token_id] * (self.max_length - len(input_ids))
        labels = self.generate_labels(input_ids)
        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

# --- Create datasets ---
config = MiniMindConfig()
pretrain_ds = PretrainDataset(pretrain_path, tokenizer, max_length=64)
sft_ds = SFTDataset(sft_path, tokenizer, max_length=128)

print(f'✅ All prior code loaded — config: {config.hidden_size}d, {config.num_hidden_layers}L, {config.vocab_size}V')
print(f'   Tokenizer: BOS={tokenizer.bos_token_id}, EOS={tokenizer.eos_token_id}, PAD={tokenizer.pad_token_id}')
print(f'   Pretrain dataset: {len(pretrain_ds)} samples, SFT dataset: {len(sft_ds)} samples')

## Part A — Optimizer & Learning Rate Schedule

We use **AdamW** (Adam with decoupled weight decay) — the standard optimizer for transformer training. For the learning rate, we use **cosine annealing with warmup**: the LR increases linearly during warmup, then decays following a cosine curve.

In [ ]:
# === Optimizer & LR Scheduler ===
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for Colab
import matplotlib.pyplot as plt

config = MiniMindConfig()
model = MiniMindForCausalLM(config).to(device)

# Optimizer
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Cosine LR scheduler with warmup
NUM_EPOCHS = 3
WARMUP_RATIO = 0.1
total_steps = NUM_EPOCHS * len(pretrain_ds) // 4  # batch_size=4
warmup_steps = int(total_steps * WARMUP_RATIO)

def get_lr(step, total_steps, warmup_steps, max_lr, min_lr=1e-6):
    """Cosine annealing with linear warmup."""
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

# Visualize the LR schedule
steps = list(range(total_steps))
lrs = [get_lr(s, total_steps, warmup_steps, LEARNING_RATE) for s in steps]
fig, ax = plt.subplots(1, 1, figsize=(8, 3))
ax.plot(steps, lrs)
ax.set_xlabel('Step')
ax.set_ylabel('Learning Rate')
ax.set_title('Cosine LR Schedule with Warmup')
ax.axvline(x=warmup_steps, color='r', linestyle='--', alpha=0.5, label=f'Warmup ends (step {warmup_steps})')
ax.legend()
plt.tight_layout()
plt.savefig('lr_schedule.png', dpi=100)
plt.show()
print(f'Total steps: {total_steps}, Warmup: {warmup_steps}')
print(f'Peak LR: {LEARNING_RATE}, Min LR: 1e-6')

## Part B — Mixed Precision Training (AMP)

Modern GPUs are much faster with float16/bfloat16 than float32. **Automatic Mixed Precision (AMP)** lets us train in lower precision where safe, and keep critical operations (like loss computation) in float32.

Key components:
- `torch.amp.autocast`: Automatically casts operations to float16/bfloat16
- `torch.amp.GradScaler`: Scales gradients to prevent underflow in float16

In [ ]:
# === Pretraining Loop ===
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 1
MAX_GRAD_NORM = 1.0
CHECKPOINT_DIR = 'minimind_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Fresh model
model = MiniMindForCausalLM(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Data
pretrain_ds = PretrainDataset(pretrain_path, tokenizer, max_length=64)
train_loader = DataLoader(pretrain_ds, batch_size=BATCH_SIZE, shuffle=True)

# AMP setup
use_amp = device == 'cuda'
dtype = torch.float32  # default for CPU
if use_amp:
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
scaler = torch.amp.GradScaler(enabled=use_amp and dtype == torch.float16)

# Training
total_steps = NUM_EPOCHS * len(train_loader)
warmup_steps = int(total_steps * WARMUP_RATIO)
loss_history = []
global_step = 0

print(f'Training config:')
print(f'  Epochs: {NUM_EPOCHS}, Batch size: {BATCH_SIZE}')
print(f'  Steps per epoch: {len(train_loader)}, Total steps: {total_steps}')
print(f'  Device: {device}, AMP: {use_amp} ({dtype if use_amp else "disabled"})')
print(f'  Parameters: {sum(p.numel() for p in model.parameters()):,}')
print()

model.train()
for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    for step, (input_ids, labels) in enumerate(train_loader):
        input_ids = input_ids.to(device)
        labels = labels.to(device)

        # Update learning rate
        lr = get_lr(global_step, total_steps, warmup_steps, LEARNING_RATE)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        # Forward pass with AMP
        with torch.amp.autocast(device_type=device, dtype=dtype, enabled=use_amp):
            outputs = model(input_ids, labels=labels)
            loss = outputs['loss'] / GRADIENT_ACCUMULATION

        # Backward pass
        if use_amp and dtype == torch.float16:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        # Optimizer step (with gradient accumulation)
        if (step + 1) % GRADIENT_ACCUMULATION == 0:
            if use_amp and dtype == torch.float16:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()
            optimizer.zero_grad()

        actual_loss = loss.item() * GRADIENT_ACCUMULATION
        epoch_loss += actual_loss
        loss_history.append(actual_loss)
        global_step += 1

        if step % 2 == 0:
            print(f'  Epoch {epoch+1}/{NUM_EPOCHS} | Step {step+1}/{len(train_loader)} | Loss: {actual_loss:.4f} | LR: {lr:.2e}')

    avg_loss = epoch_loss / len(train_loader)
    print(f'Epoch {epoch+1} complete — avg loss: {avg_loss:.4f}')

    # Save checkpoint
    ckpt_path = os.path.join(CHECKPOINT_DIR, f'pretrain_epoch{epoch+1}.pth')
    torch.save(model.state_dict(), ckpt_path)
    print(f'  Checkpoint saved → {ckpt_path}')

print(f'\n✅ Pretraining complete! Final loss: {loss_history[-1]:.4f}')

In [ ]:
# === ✅ Milestone 4A: Training Loss ===
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(loss_history, alpha=0.7)
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Pretraining Loss Curve')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=100)
plt.show()

# Verify loss decreased
initial_loss = sum(loss_history[:3]) / 3
final_loss = sum(loss_history[-3:]) / 3
print(f'Initial loss (avg first 3): {initial_loss:.4f}')
print(f'Final loss (avg last 3):    {final_loss:.4f}')
print(f'Improvement: {initial_loss - final_loss:.4f}')

assert final_loss < initial_loss, 'Loss should decrease during training!'
print(f'\n✅ Milestone 4A passed — loss decreased from {initial_loss:.4f} → {final_loss:.4f}')

## Part C — Checkpoint Loading & Verification

Checkpoints save the model's learned weights so training can be resumed or the model can be used for inference later.

In [ ]:
# === Load Checkpoint ===
# Create a fresh model and load the checkpoint
model_loaded = MiniMindForCausalLM(config).to(device)

# Loss before loading (random weights)
test_input = torch.randint(0, config.vocab_size, (1, 32)).to(device)
test_labels = torch.randint(0, config.vocab_size, (1, 32)).to(device)

with torch.no_grad():
    random_loss = model_loaded(test_input, labels=test_labels)['loss'].item()

# Load checkpoint
ckpt_path = os.path.join(CHECKPOINT_DIR, f'pretrain_epoch{NUM_EPOCHS}.pth')
model_loaded.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))

with torch.no_grad():
    loaded_loss = model_loaded(test_input, labels=test_labels)['loss'].item()

print(f'Random model loss: {random_loss:.4f}')
print(f'Loaded model loss: {loaded_loss:.4f}')
print(f'✅ Checkpoint loaded successfully — loss changed from {random_loss:.4f} → {loaded_loss:.4f}')

## Part D — Text Generation from Pretrained Model

Let's see what our pretrained model generates! Even with minimal training data, it should produce somewhat coherent text compared to random noise.

In [ ]:
# === Text Generation ===
@torch.no_grad()
def generate(model, tokenizer, prompt_ids, max_new_tokens=50, temperature=1.0, top_k=0):
    """Simple autoregressive text generation."""
    model.eval()
    input_ids = torch.tensor([prompt_ids], dtype=torch.long).to(device)

    for _ in range(max_new_tokens):
        outputs = model(input_ids)
        logits = outputs['logits'][:, -1, :] / temperature

        if top_k > 0:
            topk_values, _ = torch.topk(logits, top_k)
            # Zero out non-top-k logits so only the top-k tokens can be sampled
            logits[logits < topk_values[:, -1:]] = float('-inf')

        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_token], dim=-1)

        if next_token.item() == tokenizer.eos_token_id:
            break

    return input_ids[0].tolist()

# Generate from a few different starting prompts
model.eval()
prompts = [
    'The',
    'Language',
    'Neural',
    'Attention',
]

print('=== Generated Text ===')
for prompt_text in prompts:
    prompt_ids = tokenizer(prompt_text).input_ids
    generated_ids = generate(model, tokenizer, prompt_ids, max_new_tokens=30, temperature=0.8, top_k=50)
    generated_text = raw_tokenizer.decode(generated_ids)
    print(f'Prompt: "{prompt_text}"')
    print(f'Output: "{generated_text}"')
    print()

In [ ]:
# === ✅ Milestone 4B: Text Generation ===
prompt_ids = tokenizer('The').input_ids
generated = generate(model, tokenizer, prompt_ids, max_new_tokens=20, temperature=1.0)

assert len(generated) > len(prompt_ids), 'Model should generate new tokens'
decoded = raw_tokenizer.decode(generated)
assert len(decoded) > 0, 'Decoded text should not be empty'

print(f'✅ Milestone 4B passed — model generates text')
print(f'   Input tokens:  {len(prompt_ids)}')
print(f'   Output tokens: {len(generated)}')
print(f'   Generated: "{decoded}"')

## 🎯 Notebook 4 Summary

| Component | Key Concept |
|---|---|
| **AdamW Optimizer** | Adam with decoupled weight decay |
| **Cosine LR Schedule** | Linear warmup → cosine decay |
| **AMP** | Automatic Mixed Precision (float16/bfloat16) |
| **Gradient Clipping** | Clip gradient norm to prevent instability |
| **Checkpointing** | Save/load model state dict |
| **Text Generation** | Autoregressive decoding with temperature & top-k |

**Milestones completed:** 4A (loss decreased), 4B (text generation)

### Next Notebook
In Notebook 5, we'll do **Supervised Fine-Tuning (SFT)** to teach the model to follow instructions, and implement **LoRA** for parameter-efficient fine-tuning.